# Task1

1️ Create AstraDB account

Go to https://www.datastax.com/astra

Sign up → Astra Portal

2️ Create a Vector Database

Click Create Database

Choose:

Database type: Vector

Cloud provider: any

Region: nearest

Create database

3️ Generate Token & Endpoint

Go to Settings → Token Management

Create token with Database Administrator

Save:

ASTRA_DB_APPLICATION_TOKEN

ASTRA_DB_API_ENDPOINT

ASTRA_DB_KEYSPACE

# Task2

In [2]:
!pip install --upgrade cassio

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Windows\Temp\ipykernel_16532\3055314890.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\conda\envs\venv\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [6]:
!pip uninstall -y astrapy cassio


Found existing installation: astrapy 2.1.0
Uninstalling astrapy-2.1.0:
  Successfully uninstalled astrapy-2.1.0
Found existing installation: cassio 0.1.10
Uninstalling cassio-0.1.10:
  Successfully uninstalled cassio-0.1.10


In [7]:
!pip install astrapy==1.0.0 cassio==0.1.6


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for bson: filename=bson-0.5.10-py3-none-any.whl size=12044 sha256=d370e8e93715bd74230f0c0c3de8892e7d2531bd225ae27c842356f12098a077
  Stored in directory: c:\users\kumar\appdata\local\pip\cache\wheels\36\49\3b\8b33954dfae7a176009c4d721a45af56c8a9c1cdc3ee947945
Successfully built bson

  Attempting uninstall: uuid6

    Found existing installation: uuid6 2025.0.1

    Uninstalling uuid6-2025.0.1:

      Successfully uninstalled uuid6-2025.0.1

   ---------- ----------------------------- 1/4 [bson]
   -------------------- ------------------- 2/4 [cassio]
   -------------------- ------------------- 2/4 [cassio]
   -------------------- ---------------

In [3]:
from langchain_community.vectorstores import AstraDB

vectorstore = AstraDB(
    embedding=embeddings,
    api_endpoint=os.environ["ASTRA_DB_API_ENDPOINT"],
    token=os.environ["ASTRA_DB_APPLICATION_TOKEN"],
    namespace=os.environ["ASTRA_DB_KEYSPACE"],
    collection_name="pdf_rag_embeddings",
)

C:\Windows\Temp\ipykernel_16532\295001829.py:3: LangChainDeprecationWarning: The class `AstraDB` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the `langchain-astradb package and should be used instead. To use it run `pip install -U `langchain-astradb` and import as `from `langchain_astradb import AstraDBVectorStore``.
  vectorstore = AstraDB(


# Task3

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader

# Load PDF
loader = PyMuPDFLoader("C:/Users/kumar/OneDrive/Desktop/TRY-3/Tasks-Submission/Assignment37/LLM2024.pdf")  # <-- put your PDF here
documents = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(docs)}")


Total chunks created: 68


# Task4

In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import AstraDB
import os

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = AstraDB(
    embedding=embeddings,
    api_endpoint=os.environ["ASTRA_DB_API_ENDPOINT"],
    token=os.environ["ASTRA_DB_APPLICATION_TOKEN"],
    namespace=os.environ["ASTRA_DB_KEYSPACE"],
    collection_name="pdf_rag_embeddings",
)

vectorstore.add_documents(docs)

print("✅ Embeddings stored in AstraDB")


✅ Embeddings stored in AstraDB


In [13]:
vectorstore.similarity_search("introduction", k=1)

[Document(metadata={'producer': 'macOS Version 14.4 (Build 23E214) Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20240315052541Z00'00'", 'source': 'C:/Users/kumar/OneDrive/Desktop/TRY-3/Tasks-Submission/Assignment37/LLM2024.pdf', 'file_path': 'C:/Users/kumar/OneDrive/Desktop/TRY-3/Tasks-Submission/Assignment37/LLM2024.pdf', 'total_pages': 67, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20240315052541Z00'00'", 'trapped': '', 'modDate': "D:20240315052541Z00'00'", 'creationDate': "D:20240315052541Z00'00'", 'page': 56}, page_content='Copyright')]

# Task5

In [17]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [36]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{input}
""")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

formatted_prompt = prompt.format(
    context="LLMs are large neural networks trained on massive text corpora.",
    input="Can you tell about LLMs?"
)

response = llm.invoke(formatted_prompt)
print(response)


content='LLMs are large neural networks trained on massive text corpora.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 94, 'total_tokens': 108, 'completion_time': 0.026462849, 'completion_tokens_details': None, 'prompt_time': 0.005548767, 'prompt_tokens_details': None, 'queue_time': 0.045089252, 'total_time': 0.032011616}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019c34c6-e40c-7752-96b1-3d010593f79a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 94, 'output_tokens': 14, 'total_tokens': 108}


# Task6

In [37]:
formatted_prompt = prompt.format(
    context="LLMs are large neural networks trained on massive text corpora.",
    input="who is the prime minister of India?"
)
response2 = llm.invoke(formatted_prompt)
print(response2.content)

I don't know.


In [38]:
formatted_prompt = prompt.format(
    context="LLMs are large neural networks trained on massive text corpora.",
    input="Can you explain about Large Language models in detail?"
)
response3 = llm.invoke(formatted_prompt)
print(response3.content)

Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to process and understand human language. Here's a detailed explanation:

**What are Large Language Models?**

LLMs are large neural networks trained on massive text corpora, which are collections of text data. These models are trained to learn patterns and relationships within language, enabling them to generate human-like responses to a wide range of questions and prompts.

**How are LLMs trained?**

LLMs are trained using a process called deep learning, where the model is fed a large corpus of text data and asked to predict the next word in a sequence. This process is repeated millions of times, allowing the model to learn complex patterns and relationships within language.

**What are the key characteristics of LLMs?**

1. **Large Scale**: LLMs are trained on massive amounts of text data, often exceeding hundreds of gigabytes.
2. **Neural Network Architecture**: LLMs are based on a type of neural netwo

# Observations & Insights

1. Why AstraDB is useful for production-grade RAG applications

AstraDB is a fully managed, cloud-native vector database that provides high scalability, low-latency vector search, and built-in persistence. It eliminates operational overhead such as infrastructure management and scaling, making it well suited for production RAG systems where reliability, availability, and performance are critical.

2. Importance of session state in Generative AI applications

Session state allows a GenAI application to retain conversational context across multiple user interactions. This enables follow-up questions, improves response coherence, and provides a more natural conversational experience. In RAG systems, session state helps maintain continuity between retrieved documents and user intent.

3. Difference between FAISS and AstraDB

FAISS is a fast, local vector similarity library primarily suited for experimentation and small-scale applications. AstraDB, on the other hand, is a managed, distributed vector database that supports persistence, horizontal scaling, and production-ready deployment. While FAISS requires manual infrastructure handling, AstraDB is designed for real-world, cloud-based RAG applications.